Este cuaderno de Google Colab pretende explicar cómo hacer una transcripción de un fichero de audio usando el modelo más grande de Whisper.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Instalar Whisper y ffmpeg-python
!pip install openai-whisper ffmpeg-python

In [ ]:
import whisper
import ffmpeg
import os

# Cargar el modelo más grande de Whisper
print("Cargando el modelo 'large' de Whisper (esto puede tomar unos minutos)...")
model = whisper.load_model("large")
print("Modelo cargado correctamente.")

In [ ]:
def convertir_opus_a_wav(input_path, output_path):
    """
    Convierte un archivo .opus a .wav usando ffmpeg
    """
    try:
        ( 
            ffmpeg
            .input(input_path)
            .output(output_path, acodec='pcm_s16le', ac=1, ar='16000')
            .overwrite_output()
            .run(quiet=True)
        )
        print(f"Conversión exitosa: {output_path}")
        return True
    except ffmpeg.Error as e:
        print(f"Error al convertir el archivo: {e}")
        return False

In [ ]:
# Definir las rutas del archivo de origen y destinos
input_opus = '/content/drive/MyDrive/colabFiles/transcribirWhisper/grabacion1.opus'
output_wav = '/content/drive/MyDrive/colabFiles/transcribirWhisper/grabacion1.wav'
output_txt = '/content/drive/MyDrive/colabFiles/transcribirWhisper/grabacion1.txt'
output_srt = '/content/drive/MyDrive/colabFiles/transcribirWhisper/grabacion1.srt'

# Asegurar que el directorio de salida existe
os.makedirs(os.path.dirname(input_opus), exist_ok=True)

# 1. Convertir el archivo a .wav
exito_conversion = convertir_opus_a_wav(input_opus, output_wav)

if exito_conversion:
    # 2. Transcribir usando Whisper
    print("Iniciando transcripción con Whisper (modelo 'large')...")
    result = model.transcribe(output_wav, verbose=True)
    
    # 3. Guardar el archivo de texto plano (.txt)
    with open(output_txt, 'w', encoding='utf-8') as f:
        f.write(result['text'])
    print(f"Transcripción de texto guardada en: {output_txt}")
    
    # 4. Formatear y guardar en formato de subtítulos (.srt)
    from whisper.utils import get_writer
    writer = get_writer("srt", os.path.dirname(output_srt))
    writer(result, output_wav)
    print(f"Subtítulos SRT guardados en: {output_srt}")
else:
    print("No se pudo proceder con la transcripción debido a un fallo en la conversión del archivo de audio.")